In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable

##1 tratar schema

In [0]:
def tratar_schema(df: DataFrame) -> DataFrame:
    df_contas = df.withColumns({
        "cliente_id": col("cliente_id").cast("long")
    })

    return df_contas

###2 normalizar_strings

In [0]:
def normalizar_strings(df: DataFrame) ->DataFrame:
    df_norm = df.withColumns({
        "moeda": trim(upper(col("moeda"))),
        "iban": trim(upper(col("iban"))),
        "status": trim(upper(col("status"))),
        "tipo_conta": trim(upper(col("tipo_conta"))),
        "numero_conta": trim(col("numero_conta"))
    })

    return df_norm

###3 tratar nulls

In [0]:
def tratar_vazios(df: DataFrame) -> DataFrame:
    campos_string = [
        "numero_conta",
        "tipo_conta",
        "status",
        "iban",
        "moeda"
    ]
    df = df.withColumns({
        campo : when(col(campo) == "", None).otherwise(col(campo))
        for campo in campos_string
    })

    return df


def salvar_rejeitados(df: DataFrame, tabela: str) -> None:

    try:
        if not spark.catalog.tableExists(tabela):
            df.write.format("delta").mode("overwrite").options("mergeSchema", "True").saveAsTable(tabela)
        else:
            delta_table = DeltaTable.forName(spark, tabela)
        
            (
                delta_table.alias("destino") \
                    .merge(df.alias("destino"), "destino.id = origem.id") \
                    .whenNotMatchedInsertAll() \
                    .whenMatchedUpdateAll() \
                    .execute()
            )
    except Exception as e:
        print(f"Error: {e}")
        return False
    return True
        

In [0]:
def tratar_nulls(df: DataFrame) -> DataFrame:
    df_contas = tratar_vazios(df)
    campos_obrigatorios = [
        "id",
        "numero_conta",
        "tipo_conta",
        "saldo",
        "data_abertura",
        "status",
        "cliente_id",
        "iban",
        "moeda"
    ]

    for campos in campos_obrigatorios:
        df_rejeitados= df_contas.filter(col(campos).isNull())
        if df_rejeitados.count() > 0:
            df_rejeitados = salvar_rejeitados(df_rejeitados, "dbw_banco_orion.silver.contas_rejeitados")

    return df_contas

###Main

In [0]:
df_contas = spark.read.table("dbw_banco_orion.bronze.contas")
df_schemas = tratar_schema(df_contas)
df_norm_str = normalizar_strings(df_schemas)
df_nulos = tratar_nulls(df_norm_str)
df_nulos.select("*").summary("count").show()